# So sánh miệng MuseTalk 1.5 với InfiniteTalk trên cùng khung hình (ADR-063)

Notebook chạy **trên Kaggle**, không chạy trên laptop. MuseTalk 1.5 lồng tiếng lại đúng 81 khung InfiniteTalk (bản `preview`, version 6
của notebook `ai-host-infinitetalk-demo-host-02`) với cùng đoạn tiếng 0–3,24 giây của `preview.wav`. Kết quả là video đặt cạnh nhau để
người xem chấm miệng. Nếu MuseTalk đạt, video huấn luyện Ultralight sẽ đi đường lai: InfiniteTalk tạo nền chuyển động ngắn, MuseTalk
lồng miệng theo 3–5 phút tiếng (ADR-063 mục 3).

**Quy tắc:**
- Đây là thử nghiệm và dữ liệu huấn luyện offline, không bao giờ phát live (ràng buộc 1 của CLAUDE.md).
- Nhân vật AI `demo_host_02` chỉ dùng cho một khách, một shop.
- Video, khung hình và `run.json` **không đưa vào git**.

**Cách chạy:**
1. Add Input → **Datasets** → `jintran0501/ai-live-infinitetalk-input`, bản có cả `preview.wav` và thư mục
   `infinitetalk_v6_frames/` (81 khung `frame_*.png` lấy từ đầu ra version 6 của notebook InfiniteTalk; Kaggle không cho gắn
   đầu ra của lần chạy lỗi nên khung được đưa vào dataset).
2. Không cần gắn notebook nào khác.
3. Session options: Accelerator = **GPU T4** (một GPU là đủ), Internet = **On**.
4. Save Version → Save & Run All. Kết quả ở `musetalk_out/<RUN_NAME>/` trong panel Output.

**Xem gì:** mở `side_by_side.mp4` và `mouth_closeup.mp4` (có tiếng). Chấm theo bảng ở cuối notebook.

## 1. Cấu hình

In [ ]:
CHARACTER_ID = "demo_host_02"  # one character = one customer, one shop
RUN_NAME = "compare_v1"

# Inputs are attached in the Kaggle UI (Add Input); both are searched recursively under /kaggle/input.
AUDIO_NAME = "preview.wav"  # same audio InfiniteTalk used; its frame 0 maps to t=0 of this file
FRAME_GLOB = "frame_*.png"  # InfiniteTalk frames from the infinitetalk_out/<run>/frames/<subfolder>/ output
FPS = 25

# Official MuseTalk (MIT), pinned; only the v1.5 inference path is used.
MUSETALK_REPO = "https://github.com/TMElyralab/MuseTalk"
MUSETALK_COMMIT = "0a89dec45a0192b824e3cf4daf96c239440c5ed8"

# Hugging Face revisions pinned in ADR-063.
HF_WEIGHTS = {
    "musetalk": (
        "TMElyralab/MuseTalk",
        "2bcb936e2fddb4d86db4c62fd45b387d0c061571",
        ["musetalkV15/musetalk.json", "musetalkV15/unet.pth"],
        "",
    ),
    "sd-vae": (
        "stabilityai/sd-vae-ft-mse",
        "31f26fdeee1355a5c34592e401dd41e45d25a493",
        ["config.json", "diffusion_pytorch_model.safetensors"],
        "sd-vae",
    ),
    "whisper": (
        "openai/whisper-tiny",
        "169d4a4341b33bc18d8881c4b69c2e104e1cc0af",
        ["config.json", "model.safetensors", "preprocessor_config.json"],
        "whisper",
    ),
}
FACE_PARSE_GDRIVE_ID = "154JgKpzCPW82qINcVieuPH3fZ2e0P812"  # same source as MuseTalk's download_weights.sh
FACE_PARSE_HF_FALLBACK = ("ManyOtherFunctions/face-parse-bisent", "0073b233a5a3c4b1377d4dbf49245017938a72b5")
RESNET18_URL = "https://download.pytorch.org/models/resnet18-5c106cde.pth"
RESNET18_SHA256_PREFIX = "5c106cde"

# Defaults of the official v1.5 inference script.
BATCH_SIZE = 8
EXTRA_MARGIN = 10
LEFT_CHEEK_WIDTH = RIGHT_CHEEK_WIDTH = 90
PARSING_MODE = "jaw"
AUDIO_PADDING_LEFT = AUDIO_PADDING_RIGHT = 2
USE_FP16 = True

# Also dub the whole preview.wav, cycling the frames forward then backward like the official script.
FULL_AUDIO_RUN = True
MOUTH_CLOSEUP_PX = 360  # height of each side of the mouth close-up video

## 2. Môi trường và mã MuseTalk

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import tarfile
import time
from datetime import UTC, datetime
from pathlib import Path

import torch  # preinstalled by Kaggle with the right CUDA build

ON_KAGGLE = Path("/kaggle").exists()
if not ON_KAGGLE:
    raise SystemExit("Notebook này chỉ chạy trên Kaggle (ADR-063), không chạy trên laptop.")
if not torch.cuda.is_available():
    raise SystemExit("Chưa bật GPU: Session options → Accelerator = GPU T4.")

MUSETALK_DIR = Path("/tmp/MuseTalk")
WORK_DIR = Path("/tmp/musetalk_work")
OUT_DIR = Path("/kaggle/working/musetalk_out") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# transformers 5.x (Kaggle default) changed how encoders return hidden states, which MuseTalk's whisper path
# relies on; 4.57.6 is the last 4.x and matches the InfiniteTalk notebook. diffusers 0.30.2 is the official pin.
PIP_PINS = [
    "transformers==4.57.6",
    "diffusers==0.30.2",
    "face-alignment==1.4.1",
    "gdown",
    "einops",
    "omegaconf",
    "librosa",
]


def run(*cmd: str, cwd: Path | None = None) -> str:
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} lỗi:\n{result.stderr[-3000:]}")
    return result.stdout.strip()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


if not MUSETALK_DIR.exists():
    MUSETALK_DIR.mkdir(parents=True)
    run("git", "init", "-q", cwd=MUSETALK_DIR)
    run("git", "fetch", "-q", "--depth", "1", MUSETALK_REPO, MUSETALK_COMMIT, cwd=MUSETALK_DIR)
    run("git", "checkout", "-q", "FETCH_HEAD", cwd=MUSETALK_DIR)
assert run("git", "rev-parse", "HEAD", cwd=MUSETALK_DIR) == MUSETALK_COMMIT

# Keep Kaggle's CUDA build of torch: none of the pins below pull torch.
run(sys.executable, "-m", "pip", "install", "-q", *PIP_PINS)
run("ffmpeg", "-version")

import diffusers  # noqa: E402
import face_alignment  # noqa: E402
import transformers  # noqa: E402

GPU_NAME = torch.cuda.get_device_name(0)
print(
    f"GPU: {GPU_NAME} · torch {torch.__version__} · transformers {transformers.__version__} · "
    f"diffusers {diffusers.__version__} · face-alignment {face_alignment.__version__} · "
    f"MuseTalk {MUSETALK_COMMIT[:10]}"
)

## 3. Trọng số (ghim revision, ghi sha256)

In [ ]:
from huggingface_hub import hf_hub_download  # noqa: E402

MODELS_DIR = MUSETALK_DIR / "models"
weights_info: dict[str, dict] = {}

for key, (repo, revision, files, subdir) in HF_WEIGHTS.items():
    target = MODELS_DIR / subdir if subdir else MODELS_DIR
    for name in files:
        started = time.time()
        path = Path(hf_hub_download(repo, name, revision=revision, local_dir=target))
        weights_info[f"{key}/{name}"] = {"repo": repo, "revision": revision, "sha256": sha256(path)}
        print(f"{repo}/{name} @ {revision[:10]} ({time.time() - started:.0f} s)")

face_parse_dir = MODELS_DIR / "face-parse-bisent"
face_parse_dir.mkdir(parents=True, exist_ok=True)
face_parse_path = face_parse_dir / "79999_iter.pth"
face_parse_source = f"gdrive:{FACE_PARSE_GDRIVE_ID}"
if not face_parse_path.exists():
    try:
        run(sys.executable, "-m", "gdown", FACE_PARSE_GDRIVE_ID, "-O", str(face_parse_path))
    except RuntimeError as error:
        # Google Drive often rate-limits shared files; fall back to the pinned HF copy.
        print(f"gdown lỗi, dùng bản sao HF: {str(error)[-300:]}")
        repo, revision = FACE_PARSE_HF_FALLBACK
        hf_hub_download(repo, "79999_iter.pth", revision=revision, local_dir=face_parse_dir)
        face_parse_source = f"hf:{repo}@{revision}"
weights_info["face-parse/79999_iter.pth"] = {"source": face_parse_source, "sha256": sha256(face_parse_path)}

resnet_path = face_parse_dir / "resnet18-5c106cde.pth"
if not resnet_path.exists():
    # torchvision names weight files after their sha256 prefix; download_url_to_file rejects a mismatch.
    torch.hub.download_url_to_file(RESNET18_URL, str(resnet_path), hash_prefix=RESNET18_SHA256_PREFIX)
weights_info["face-parse/resnet18-5c106cde.pth"] = {"source": RESNET18_URL, "sha256": sha256(resnet_path)}

# The 2017 torchvision ResNet-18 file uses the legacy tar format, which torch>=2.6 cannot open with weights_only and
# which MuseTalk's FaceParsing loads with torch's default (weights_only=True). Its sha256 prefix is checked above and
# it comes from download.pytorch.org, so it is unpickled once and re-saved as a plain state dict in the zip format.
if tarfile.is_tarfile(resnet_path):
    state_dict = torch.load(resnet_path, map_location="cpu", weights_only=False)
    torch.save({key: value for key, value in state_dict.items()}, resnet_path)
    weights_info["face-parse/resnet18-5c106cde.pth"]["converted_sha256"] = sha256(resnet_path)
    print("resnet18-5c106cde.pth: định dạng tar cũ, đã lưu lại dạng zip (state dict).")

# Every other checkpoint must be a plain state dict: refuse anything that needs pickle code execution.
for path in (face_parse_path, resnet_path, MODELS_DIR / "musetalkV15" / "unet.pth"):
    torch.load(path, map_location="cpu", weights_only=True)
print("Đủ trọng số; các file .pth nạp được ở chế độ weights_only.")

## 4. Đầu vào: 81 khung InfiniteTalk và đoạn tiếng tương ứng

In [ ]:
import cv2  # noqa: E402
import numpy as np  # noqa: E402

INPUT_ROOT = Path("/kaggle/input")
frame_paths = sorted(INPUT_ROOT.rglob(FRAME_GLOB))
audio_candidates = sorted(INPUT_ROOT.rglob(AUDIO_NAME))
if not frame_paths:
    raise SystemExit(f"Không thấy {FRAME_GLOB}: thiếu thư mục infinitetalk_v6_frames/ trong dataset.")
if not audio_candidates:
    raise SystemExit(f"Không thấy {AUDIO_NAME}: gắn dataset jintran0501/ai-live-infinitetalk-input.")
frame_dirs = sorted({path.parent for path in frame_paths})
if len(frame_dirs) > 1:
    print(f"Có {len(frame_dirs)} thư mục khung, dùng thư mục đầu: {frame_dirs[0]}")
frame_paths = sorted(frame_dirs[0].glob(FRAME_GLOB))
source_audio = audio_candidates[0]

frames = [cv2.imread(str(path)) for path in frame_paths]
height, width = frames[0].shape[:2]
n_frames = len(frames)
clip_seconds = n_frames / FPS


def probe_seconds(path: Path) -> float:
    return float(run("ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(path)))


# A little extra audio so floor(seconds * fps) never lands one frame short; extra chunks are dropped later.
clip_audio = WORK_DIR / "clip.wav"
run(
    "ffmpeg",
    "-y",
    "-v",
    "error",
    "-i",
    str(source_audio),
    "-t",
    f"{clip_seconds + 0.04:.3f}",
    "-ac",
    "1",
    "-ar",
    "16000",
    str(clip_audio),
)
full_audio = WORK_DIR / "full.wav"
run("ffmpeg", "-y", "-v", "error", "-i", str(source_audio), "-ac", "1", "-ar", "16000", str(full_audio))
print(f"Khung: {n_frames} ({width}×{height}) từ {frame_dirs[0]}")
print(f"Tiếng: {source_audio} ({probe_seconds(full_audio):.2f} s); đoạn so sánh 0–{clip_seconds:.2f} s")

## 5. Khung cắt mặt

Script chính thức lấy 68 điểm mặt bằng DWPose (cần mmcv/mmpose). Ở đây dùng gói `face-alignment`: cùng 68 điểm theo thứ tự iBUG, nên công
thức khung cắt giữ y như `get_landmark_and_bbox` của MuseTalk (ADR-063). Ảnh `crop_check.png` vẽ khung cắt và điểm mặt để kiểm.

In [ ]:
fa_types = face_alignment.LandmarksType
fa = face_alignment.FaceAlignment(getattr(fa_types, "TWO_D", None) or fa_types._2D, flip_input=False, device="cuda")


def musetalk_bbox(landmarks: np.ndarray, detection: np.ndarray) -> tuple[int, int, int, int]:
    """Same box rule as MuseTalk's get_landmark_and_bbox with bbox_shift=0 (v1.5)."""
    lm = landmarks.astype(np.int32)
    half_face_y = lm[29][1]
    half_face_dist = np.max(lm[:, 1]) - half_face_y
    upper_bond = max(0, half_face_y - half_face_dist)
    x1, y1, x2, y2 = int(np.min(lm[:, 0])), int(upper_bond), int(np.max(lm[:, 0])), int(np.max(lm[:, 1]))
    if y2 - y1 <= 0 or x2 - x1 <= 0 or x1 < 0:
        # MuseTalk falls back to the detector box when the landmark box is unusable.
        return tuple(int(v) for v in detection[:4])
    return x1, y1, x2, y2


coords: list[tuple[int, int, int, int]] = []
landmarks_all: list[np.ndarray] = []
started = time.time()
for index, frame in enumerate(frames):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = fa.get_landmarks_from_image(rgb, return_bboxes=True)
    landmarks, _, detections = result if isinstance(result, tuple) else (result, None, None)
    if not landmarks:
        raise SystemExit(f"Khung {index}: không thấy mặt.")
    # The host is the only face; take the largest detection if a poster or product label fools the detector.
    best = max(range(len(landmarks)), key=lambda i: np.ptp(landmarks[i][:, 0]) * np.ptp(landmarks[i][:, 1]))
    landmarks_all.append(landmarks[best])
    coords.append(musetalk_bbox(landmarks[best], detections[best]))
landmark_seconds = time.time() - started

check = frames[0].copy()
x1, y1, x2, y2 = coords[0]
cv2.rectangle(check, (x1, y1), (x2, min(y2 + EXTRA_MARGIN, height)), (0, 255, 0), 2)
for x, y in landmarks_all[0].astype(int):
    cv2.circle(check, (int(x), int(y)), 2, (0, 0, 255), -1)
cv2.imwrite(str(OUT_DIR / "crop_check.png"), check)
widths = [c[2] - c[0] for c in coords]
print(
    f"Điểm mặt {n_frames} khung trong {landmark_seconds:.1f} s; rộng khung cắt {min(widths)}–{max(widths)} px "
    f"(MuseTalk phóng lên 256 px)"
)

## 6. Lồng miệng bằng MuseTalk 1.5

In [ ]:
os.chdir(MUSETALK_DIR)  # MuseTalk loads models/ by relative path
sys.path.insert(0, str(MUSETALK_DIR))

from musetalk.utils.audio_processor import AudioProcessor  # noqa: E402
from musetalk.utils.blending import get_image  # noqa: E402
from musetalk.utils.face_parsing import FaceParsing  # noqa: E402
from musetalk.utils.utils import datagen, load_all_model  # noqa: E402
from transformers import WhisperModel  # noqa: E402

device = torch.device("cuda")
vae, unet, pe = load_all_model(
    unet_model_path=str(MODELS_DIR / "musetalkV15" / "unet.pth"),
    vae_type="sd-vae",
    unet_config=str(MODELS_DIR / "musetalkV15" / "musetalk.json"),
    device=device,
)
if USE_FP16:
    pe = pe.half()
    vae.vae = vae.vae.half()
    unet.model = unet.model.half()
pe = pe.to(device)
vae.vae = vae.vae.to(device)
unet.model = unet.model.to(device)
weight_dtype = unet.model.dtype
timesteps = torch.tensor([0], device=device)

audio_processor = AudioProcessor(feature_extractor_path=str(MODELS_DIR / "whisper"))
whisper = WhisperModel.from_pretrained(str(MODELS_DIR / "whisper")).to(device=device, dtype=weight_dtype).eval()
whisper.requires_grad_(False)
face_parser = FaceParsing(left_cheek_width=LEFT_CHEEK_WIDTH, right_cheek_width=RIGHT_CHEEK_WIDTH)


def crop_box(coord: tuple[int, int, int, int]) -> tuple[int, int, int, int]:
    x1, y1, x2, y2 = coord
    return x1, y1, x2, min(y2 + EXTRA_MARGIN, height)  # v1.5 extends the chin margin


with torch.no_grad():
    input_latents = []
    for coord, frame in zip(coords, frames, strict=True):
        x1, y1, x2, y2 = crop_box(coord)
        crop = cv2.resize(frame[y1:y2, x1:x2], (256, 256), interpolation=cv2.INTER_LANCZOS4)
        input_latents.append(vae.get_latents_for_unet(crop))


@torch.no_grad()
def dub(audio_path: Path, max_frames: int | None, cycle: bool) -> tuple[list[np.ndarray], dict]:
    """Return blended frames for audio_path; cycle=True plays frames forward then backward (official script)."""
    order = list(range(n_frames)) + (list(range(n_frames))[::-1] if cycle else [])
    features, librosa_length = audio_processor.get_audio_feature(str(audio_path))
    chunks = audio_processor.get_whisper_chunk(
        features,
        device,
        weight_dtype,
        whisper,
        librosa_length,
        fps=FPS,
        audio_padding_length_left=AUDIO_PADDING_LEFT,
        audio_padding_length_right=AUDIO_PADDING_RIGHT,
    )
    if max_frames is not None:
        if len(chunks) < max_frames:
            raise SystemExit(f"Tiếng chỉ đủ {len(chunks)} khung, cần {max_frames}.")
        chunks = chunks[:max_frames]
    latents = [input_latents[i] for i in order]

    torch.cuda.synchronize()
    started = time.time()
    faces = []
    for whisper_batch, latent_batch in datagen(chunks, latents, batch_size=BATCH_SIZE, delay_frame=0, device=device):
        audio_features = pe(whisper_batch.to(device))
        prediction = unet.model(
            latent_batch.to(device, dtype=weight_dtype), timesteps, encoder_hidden_states=audio_features
        ).sample
        faces.extend(vae.decode_latents(prediction))
    torch.cuda.synchronize()
    model_seconds = time.time() - started

    started = time.time()
    out_frames = []
    for i, face in enumerate(faces):
        src = order[i % len(order)]
        x1, y1, x2, y2 = crop_box(coords[src])
        face = cv2.resize(face.astype(np.uint8), (x2 - x1, y2 - y1))
        out_frames.append(get_image(frames[src].copy(), face, [x1, y1, x2, y2], mode=PARSING_MODE, fp=face_parser))
    blend_seconds = time.time() - started
    stats = {
        "frames": len(out_frames),
        "model_seconds": round(model_seconds, 2),
        "blend_seconds": round(blend_seconds, 2),
        "model_fps": round(len(out_frames) / model_seconds, 2),
        "end_to_end_fps": round(len(out_frames) / (model_seconds + blend_seconds), 2),
        "source_order": [order[i % len(order)] for i in range(len(out_frames))],
    }
    return out_frames, stats


clip_frames, clip_stats = dub(clip_audio, n_frames, cycle=False)
print(
    f"Đoạn so sánh: {clip_stats['frames']} khung, UNet+VAE {clip_stats['model_fps']} khung/s, "
    f"tính cả ghép mặt {clip_stats['end_to_end_fps']} khung/s"
)
full_frames, full_stats = dub(full_audio, None, cycle=True) if FULL_AUDIO_RUN else ([], {})
if full_frames:
    print(f"Cả file tiếng: {full_stats['frames']} khung, {full_stats['end_to_end_fps']} khung/s")
print(f"VRAM đỉnh: {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")

## 7. Video so sánh, cận cảnh miệng, `run.json`

In [ ]:
os.chdir("/kaggle/working")
FONT = cv2.FONT_HERSHEY_SIMPLEX


def label(image: np.ndarray, text: str) -> np.ndarray:
    image = image.copy()
    cv2.putText(image, text, (12, 34), FONT, 0.9, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(image, text, (12, 34), FONT, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    return image


def write_video(path: Path, images: list[np.ndarray], audio: Path, seconds: float) -> None:
    frame_dir = WORK_DIR / f"{path.stem}_frames"
    frame_dir.mkdir(exist_ok=True)
    for old in frame_dir.glob("*.png"):
        old.unlink()
    for i, image in enumerate(images):
        cv2.imwrite(str(frame_dir / f"{i:06d}.png"), image)
    run(
        "ffmpeg",
        "-y",
        "-v",
        "error",
        "-framerate",
        str(FPS),
        "-i",
        str(frame_dir / "%06d.png"),
        "-i",
        str(audio),
        "-t",
        f"{seconds:.3f}",
        "-vf",
        "pad=ceil(iw/2)*2:ceil(ih/2)*2",
        "-c:v",
        "libx264",
        "-crf",
        "16",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-b:a",
        "128k",
        str(path),
    )


# One fixed mouth window from the median landmarks so both sides show exactly the same pixels.
mouth_points = np.concatenate([lm[48:68] for lm in landmarks_all])
center = np.median(mouth_points, axis=0)
mouth_width = float(np.median([np.ptp(lm[48:68, 0]) for lm in landmarks_all]))
half = int(mouth_width * 0.9)
mx1, my1 = max(0, int(center[0]) - half), max(0, int(center[1]) - int(half * 0.75))
mx2, my2 = min(width, int(center[0]) + half), min(height, int(center[1]) + int(half * 0.75))


def mouth(image: np.ndarray) -> np.ndarray:
    crop = image[my1:my2, mx1:mx2]
    scale = MOUTH_CLOSEUP_PX / crop.shape[0]
    return cv2.resize(crop, (int(crop.shape[1] * scale), MOUTH_CLOSEUP_PX), interpolation=cv2.INTER_CUBIC)


write_video(OUT_DIR / "infinitetalk.mp4", frames, clip_audio, clip_seconds)
write_video(OUT_DIR / "musetalk_clip.mp4", clip_frames, clip_audio, clip_seconds)
side = [
    np.hstack([label(a, "InfiniteTalk"), label(b, "MuseTalk 1.5")]) for a, b in zip(frames, clip_frames, strict=True)
]
write_video(OUT_DIR / "side_by_side.mp4", side, clip_audio, clip_seconds)
close = [
    np.hstack([label(mouth(a), "InfiniteTalk"), label(mouth(b), "MuseTalk 1.5")])
    for a, b in zip(frames, clip_frames, strict=True)
]
write_video(OUT_DIR / "mouth_closeup.mp4", close, clip_audio, clip_seconds)
if full_frames:
    write_video(OUT_DIR / "musetalk_full.mp4", full_frames, full_audio, full_stats["frames"] / FPS)

# Contact sheet: every 4th frame, InfiniteTalk row over MuseTalk row, for a quick look without playing video.
picks = list(range(0, n_frames, 4))
tiles_it = [cv2.resize(mouth(frames[i]), None, fx=0.5, fy=0.5) for i in picks]
tiles_mt = [cv2.resize(mouth(clip_frames[i]), None, fx=0.5, fy=0.5) for i in picks]
rows = []
for start in range(0, len(picks), 7):
    top = tiles_it[start : start + 7]
    bottom = tiles_mt[start : start + 7]
    while len(top) < 7:
        top.append(np.zeros_like(tiles_it[0]))
        bottom.append(np.zeros_like(tiles_it[0]))
    for j, i in enumerate(picks[start : start + 7]):
        cv2.putText(top[j], f"IT {i}", (6, 22), FONT, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(bottom[j], f"MT {i}", (6, 22), FONT, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
    rows.append(np.vstack([np.hstack(top), np.hstack(bottom), np.full((8, np.hstack(top).shape[1], 3), 255, np.uint8)]))
cv2.imwrite(str(OUT_DIR / "mouth_sheet.png"), np.vstack(rows))

for stats in (clip_stats, full_stats):
    stats.pop("source_order", None)
run_info = {
    "adr": "ADR-063",
    "character_id": CHARACTER_ID,
    "run_name": RUN_NAME,
    "created_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "gpu": GPU_NAME,
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "diffusers": diffusers.__version__,
        "face_alignment": face_alignment.__version__,
        "musetalk_commit": MUSETALK_COMMIT,
    },
    "weights": weights_info,
    "inputs": {
        "frames_dir": str(frame_dirs[0]),
        "frames": n_frames,
        "frame_size": [width, height],
        "audio": str(source_audio),
        "audio_sha256": sha256(source_audio),
        "clip_seconds": clip_seconds,
    },
    "settings": {
        "fps": FPS,
        "batch_size": BATCH_SIZE,
        "extra_margin": EXTRA_MARGIN,
        "cheek_width": [LEFT_CHEEK_WIDTH, RIGHT_CHEEK_WIDTH],
        "parsing_mode": PARSING_MODE,
        "audio_padding": [AUDIO_PADDING_LEFT, AUDIO_PADDING_RIGHT],
        "fp16": USE_FP16,
        "landmarks": "face-alignment 68 pts (replaces DWPose)",
    },
    "crop_width_px": [min(widths), max(widths)],
    "mouth_window": [mx1, my1, mx2, my2],
    "timing": {
        "landmarks_seconds": round(landmark_seconds, 1),
        "clip": clip_stats,
        "full": full_stats,
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    },
}
(OUT_DIR / "run.json").write_text(json.dumps(run_info, ensure_ascii=False, indent=2))
for path in sorted(OUT_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 2**20:.1f} MB")

## 8. Cách chấm

Xem `side_by_side.mp4` và `mouth_closeup.mp4` có tiếng, rồi `mouth_sheet.png` (hàng trên InfiniteTalk, hàng dưới MuseTalk). Với mỗi
dòng, ghi bên nào tốt hơn hoặc "ngang nhau":

| Tiêu chí | Cần thấy |
|---|---|
| Môi khép ở m, b, p | Hai môi chạm nhau rõ ràng, không chỉ hé |
| Môi tròn ở ô, u, o | Môi chu lại, không chỉ mở dọc |
| Mở rộng ở a | Hàm hạ đủ, thấy răng trên hợp lý |
| Răng, lưỡi | Không nhòe thành một mảng trắng |
| Độ nét vùng miệng | Không mờ hơn hẳn phần mặt còn lại |
| Đường nối | Không thấy viền quanh cằm hoặc má khi đầu nghiêng |
| Rung | Miệng không giật giữa các khung |
| Khớp tiếng | Miệng không đi trước hoặc trễ sau tiếng |

MuseTalk **đạt** khi bằng hoặc hơn InfiniteTalk ở phần lớn tiêu chí và không có lỗi nặng (đường nối rõ, mờ hẳn). Khi đó làm tiếp đường
lai ở ADR-063 mục 3. `musetalk_full.mp4` cho xem MuseTalk nói đủ 13,5 giây trên khung chạy tới rồi lùi.